In [ ]:

import os
import re
import string
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

#=======preprocessing=====
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize

#======Model and Feature========
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, GridSearchCV

#==========Training and evaluation===========
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,confusion_matrix, classification_report
import joblib

nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')

In [ ]:
def load_imdb_data(base_path):
    data =[]

    for split in ['train','test']:
        for sentiment in ['pos','neg']:
            folder_path = os.path.join(base_path,split,sentiment)
            for filename in os.listdir(folder_path):
                if filename.endswith('.txt'):
                    file_path = os.path.join(folder_path,filename)
                    with open (file_path, encoding='utf-8') as f:
                        review_text = f.read()
                    data.append({
                        'review': review_text,
                        'sentiment': sentiment,
                        'split': split
                    })
    return pd.DataFrame(data)
Data_path = r"C:\Users\Asus\OneDrive - K L University\Desktop\Sentiment_Analysis\aclImdb"
df = load_imdb_data(Data_path)
print("Dataset loaded successfully")
print(f"Shape:{df.shape} ")

In [ ]:
print(df['sentiment'].value_counts())
sns.countplot(data=df,x='sentiment')
plt.title("Class Distribution +ve vs -ve reviews")
plt.show()

In [ ]:
print("Positive Review Example:\n",df[df['sentiment']=='pos']['review'].iloc[18])
print("Negative Review Example:\n",df[df['sentiment']=='neg']['review'].iloc[6])

In [ ]:
df['review_length'] = df['review'].apply(lambda x: len(x.split()))
print(df['review_length'].describe())
plt.figure(figsize=(8,5))
sns.histplot(df['review_length'], bins = 50, kde=True)
plt.title('Review length distribution')
plt.xlabel("Number of words")
plt.ylabel('Frequency')
plt.show()

In [ ]:
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def clean_text(text,use_stemming= False):

    text = text.lower()
    text = re.sub(r'<.*?>',' ',text )
    text = re.sub(r'http\S+|www\S+',' ', text)
    text = re.sub(r'[^a-z\s]',' ', text)
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in stop_words and len(t)>2]

    if use_stemming:
        tokens = [stemmer.stem(t) for t in tokens]
    else:
        tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return ' '.join(tokens)

In [ ]:
sample_raw = df['review'].iloc[0]
sample_cleaned = clean_text(sample_raw)

print("Raw:\n",sample_raw[:500])
print("\n Clean:\n",sample_cleaned[:500])

In [ ]:
print("Cleaning will take a moment of your time. So, Please sit back and relax until then")
df['cleaned_review'] = df['review'].apply(clean_text)
print("\n Sample cleaned review:\n", df['cleaned_review'].iloc[1][:300])

In [ ]:
print(df['cleaned_review'].isnull().sum())
print((df['cleaned_review']=='').sum())
df['cleaned_length'] = df['cleaned_review'].apply(lambda x:len(x.split()))
print("\n Original review length stats:")
print(df['review_length'].describe())
print("\n Cleaned review length stats:")
print(df['cleaned_length'].describe())


In [ ]:
#=========Converting labels  to binary numbers==============
df['label'] = df['sentiment'].map({'pos':1,'neg':0})
print("Label Distribution:")
print(df['label'].value_counts())
print("\n Sample Rows:")
print(df[['sentiment','label']].head(10))


In [ ]:
#=============Splitting data into Training and test sets==========
X = df['cleaned_review']
y = df['label']

X_train, X_test, y_train,y_test = train_test_split(X,y,test_size = 0.2,random_state= 42, stratify = y)

print(f"Training Samples: {len(X_train)}")
print(f"Test Samples: {len(X_test)}")
print(f"\n Training label distribution:\n{y_train.value_counts()}")
print(f"\n Test label distribution:\n{y_test.value_counts()}")

In [ ]:
tfidf = TfidfVectorizer(
    max_features= 50000,
    ngram_range = (1,2),
    min_df= 3,
    max_df = 0.90,
    sublinear_tf = True
)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("TF-IDF matrix shape(train):",X_train_tfidf.shape)
print("TF-IDF matrix shape(test):",X_test_tfidf.shape)
print("\n Matrix is sparse - non-zero elements:",X_train_tfidf.nnz)
print("Sparsity: {:.2f}%".format(
    100* (1 - X_train_tfidf.nnz/ (X_train_tfidf.shape[0] * X_train_tfidf.shape[1] ) )
))


In [ ]:
# ===========Inspect the vocabulary==============

feature_names = tfidf.get_feature_names_out()
print(f"Total features (vocabulary size): {len(feature_names)}")
print("\nFirst 20 features (alphabetical):", feature_names[:20])
print("\nSome bigram examples:")
bigrams= [f for f in feature_names if ' ' in f]
print(bigrams[:20])

sample_vector = X_train_tfidf[0].toarray()[0]
top_indices= sample_vector.argsort()[-15:][::-1]
print("\nTop 15 TF-IDF features for first training review:")
for idx in top_indices:
    print(f" '{feature_names[idx]}' -> score: {sample_vector[idx]:.4f} ")

In [ ]:
#================Baseline SVM with linear Kernel=============

svm_linear = SVC(
    kernel = 'linear',
    C=1.0,
    random_state = 42,
    verbose = True
)
print("Training Linear SVM")
print("Training makes a model optimal so wait for few minutes it's completely normal....")

import time
start = time.time()
svm_linear.fit(X_train_tfidf,y_train)
end = time.time()

print(f"\n You waited for {end - start:.1f} seconds")
print(f"Number of support vectors: {svm_linear.n_support_} ")
print(f"Total Support Vectors : {sum(svm_linear.n_support_)}")

In [ ]:
y_pred_linear = svm_linear.predict(X_test_tfidf)

acc= accuracy_score(y_test,y_pred_linear)
prec = precision_score(y_test,y_pred_linear)
rec = recall_score(y_test,y_pred_linear)
f1 = f1_score(y_test,y_pred_linear)

print("Linear SVM - Evaluation results")
print(f"\n Accuracy: {acc:.4f} ({acc*100:.2f}%) ")
print(f" Precision : {prec:.4f} ")
print(f" Recall : {rec:.4f} ")
print(f" F1-Score : {f1:.4f} ")
print("\nFull Classification Report : ")
print(classification_report(y_test,y_pred_linear, target_names=['Negative','Positive']))

In [ ]:
cm = confusion_matrix(y_test, y_pred_linear)
plt.figure(figsize=(7,5))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='ocean',
    xticklabels = ['Predicted Negative', 'Predicted Positive'],
    yticklabels = ['Actual Negative', 'Actual Positive']
)

plt.title("Confusion Matrix - Linear SVM")
plt.tight_layout()
plt.show

tn,fp,fn,tp = cm.ravel()

print(f"\nTrue Negatives  (correctly predicted negative): {tn}")
print(f"False Positives (predicted positive, was negative): {fp}")
print(f"False Negatives (predicted negative, was positive): {fn}")
print(f"True Positives  (correctly predicted positive): {tp}")

In [ ]:
import time

results = {}
#==============Kernel configs to loop over=======================

kernels = [
    {
    'name' : 'Linear',
    'kernel' : 'linear',
    'params' : {'C':10}
    },
    {
    'name' : 'Polynomial',
    'kernel' : 'poly',
    'params' : {'C': 1.0, 'degree' : 3, 'coef0': 1}
    },
    {
    'name' : 'RBF',
    'kernel' : 'rbf',
    'params' : {'C': 1.0,'gamma':'scale'}
    },
]

for config in kernels:
    print(f"Training {config['name']} kernel SVM... ")
    model = SVC(
        kernel = config['kernel'],
        random_state=42,
        **config['params']
    )
    t0 = time.time()
    model.fit(X_train_tfidf,y_train)
    train_time = time.time() - t0

    t1 = time.time()
    y_pred = model.predict(X_test_tfidf)
    pred_time= time.time() -t1

    results[config['name']] = {
        'model' : model,
        'predictions' : y_pred,
        'accuracy' : accuracy_score(y_test,y_pred),
        'precision' : precision_score(y_test,y_pred),
        'recall' : recall_score(y_test,y_pred),
        'f1' : f1_score(y_test,y_pred),
        'train_time' : train_time,
        'pred_time' : pred_time,
        'n_sv' : sum(model.n_support_)
    }
    print(f" Done in {train_time:.1f}s  |"
         f"Accuracy: {results[config['name']]['accuracy']:.4f} | "
          f" F1: {results[config['name']]['f1']:.4f}  | "  
            f"SVs: {results[config['name']]['n_sv']}")